# Lab Kilat Sesi 1 — Sinyal Hari Ini, Angka Resmi Berbulan-bulan Lagi

**Introduction to Data Science for Economists** · Mikro Kredensial Luhung · Universitas Padjadjaran
Prof. Mohamad Fahmi

---

Di email selamat datang, kamu dijanjikan satu hal: melihat sendiri bagaimana data pencarian
Google bergerak jauh sebelum angka pengangguran resmi terbit. Ini notebooknya.

Kita pakai dua kata kunci. **PHK**, yang diketik orang ketika mendengar kabar pemutusan
hubungan kerja. Dan **lowongan kerja**, yang diketik orang ketika sedang mencari pekerjaan.
Keduanya terdengar sama-sama masuk akal sebagai penanda kondisi pasar kerja. Nanti kamu lihat
sendiri hanya satu yang bekerja, dan alasannya jauh lebih menarik daripada grafiknya.

**Waktu:** sekitar 30 menit. **Prasyarat:** tidak ada. Kamu tidak perlu bisa Python.

> **Cara pakai:** klik `File → Save a copy in Drive` dulu, pakai akun Google Unpad kamu.
> Kalau tidak, perubahanmu tidak tersimpan.

## Langkah 1 — Siapkan alat

Jalankan sel di bawah. Klik sel, lalu tekan `Shift + Enter`. Tunggu sampai muncul tanda centang.

Kode ini memasang satu paket bernama `pytrends` yang menjadi jembatan ke Google Trends, lalu
memanggil paket standar untuk mengolah tabel (`pandas`) dan menggambar grafik (`matplotlib`).

In [ ]:
!pip install pytrends --quiet

import warnings
import pandas as pd
import matplotlib.pyplot as plt

# pytrends memakai cara lama memanggil pandas, sehingga memunculkan peringatan
# FutureWarning yang panjang. Peringatan itu tidak memengaruhi hasil, jadi kita diamkan
# agar keluaran notebook tetap bersih dan tidak menakut-nakuti.
warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["font.size"] = 11

print("Siap.")

## Langkah 2 — Angka resmi BPS

Tabel di bawah berisi rilis Tingkat Pengangguran Terbuka nasional beserta **tanggal terbitnya**.
Perhatikan kolom terakhir: jarak antara kondisi yang diukur dan saat angkanya diketahui publik.

Sumber setiap baris ada di kolom `sumber`. Ini kebiasaan yang wajib kamu bawa sepanjang
semester: setiap angka punya alamat asal yang bisa diperiksa orang lain.

In [ ]:
# Rilis TPT nasional, diverifikasi dari Berita Resmi Statistik BPS.
bps = pd.DataFrame([
    # periode data,  tanggal rilis, TPT (%), sumber
    ("2020-08-01", "2020-11-05", 7.07, "bps.go.id/pressrelease/2020/11/05/1673"),
    ("2022-02-01", "2022-05-09", 5.83, "bps.go.id/pressrelease/2022/05/09/1915"),
    ("2024-02-01", "2024-05-06", 4.82, "bps.go.id/pressrelease/2024/05/06/2372"),
    ("2025-08-01", "2025-11-05", 4.85, "bps.go.id/pressrelease/2025/11/05/2479"),
    ("2026-02-01", "2026-05-05", 4.68, "bps.go.id/pressrelease/2026/05/05/2574"),
], columns=["periode_data", "tanggal_rilis", "tpt_persen", "sumber"])

bps["periode_data"]  = pd.to_datetime(bps["periode_data"])
bps["tanggal_rilis"] = pd.to_datetime(bps["tanggal_rilis"])
bps["jeda_hari"]     = (bps["tanggal_rilis"] - bps["periode_data"]).dt.days

display(bps[["periode_data", "tanggal_rilis", "tpt_persen", "jeda_hari"]])
print(f"\nRata-rata jeda dari periode data ke tanggal rilis: {bps['jeda_hari'].mean():.0f} hari "
      f"({bps['jeda_hari'].mean()/30:.1f} bulan)")

**Berhenti sebentar dan pikirkan.**

Kolom `jeda_hari` menunjukkan sekitar 95 hari, atau tiga bulan, antara periode yang diukur dan
tanggal terbitnya. Itu jeda administratifnya.

Tapi jeda yang benar-benar dirasakan pembuat kebijakan lebih panjang lagi. Pasar kerja Indonesia
runtuh pada April 2020. Kondisi itu baru tertangkap survei pada Agustus 2020, dan angkanya baru
diumumkan 5 November 2020. Dari awal gelombang PHK sampai angka resminya terbit: tujuh bulan.

Selama tujuh bulan itu pemerintah harus memutuskan besaran bantuan, sasarannya, dan lamanya,
tanpa satu pun angka resmi yang menunjukkan seberapa parah keadaannya.

Itu bukan kelalaian BPS. Survei Angkatan Kerja Nasional melibatkan ratusan ribu rumah tangga,
dan jedanya melekat pada metodenya. Pertanyaannya bukan bagaimana menghapus jeda itu, melainkan
apa yang bisa dipakai untuk mengisinya.

## Langkah 3 — Sinyal dari mesin pencari

Sekarang kita ambil data yang tersedia hari ini juga.

Angkanya bukan jumlah orang. Google Trends memberi **indeks relatif 0 sampai 100**, di mana
100 adalah titik tertinggi dalam periode, wilayah, dan **daftar kata kunci** yang kamu minta.
Tiga hal terakhir itu penting, dan sebentar lagi akan menggigit kita.

> **Kalau muncul pesan merah panjang bertuliskan `ModuleNotFoundError`,** berarti sel Langkah 1
> belum dijalankan. Jalankan Langkah 1 dulu, lalu ulangi sel ini. Cara paling aman untuk seluruh
> notebook: menu `Runtime → Run all`.

In [ ]:
KATA    = ["PHK", "lowongan kerja"]          # <-- nanti kamu ganti
PERIODE = "2019-01-01 2026-08-23"            # <-- nanti kamu ganti
WILAYAH = "ID"                                # ID = Indonesia

# CATATAN UNTUK DOSEN: ubah ke True sekali saja untuk menguji jalur cadangan,
# lalu kembalikan ke False sebelum sesi dibuka.
PAKSA_CADANGAN = False

trends = None

if PAKSA_CADANGAN:
    print("Mode uji aktif. Google Trends sengaja dilewati.")
    print("Jalankan sel berikutnya untuk menguji salinan cadangan.")
else:
    try:
        # Import diletakkan di dalam try, supaya notebook tidak berhenti total
        # kalau paket belum terpasang atau runtime baru saja di-restart.
        from pytrends.request import TrendReq

        pt = TrendReq(hl="id-ID", tz=420)
        pt.build_payload(KATA, timeframe=PERIODE, geo=WILAYAH)
        hasil = pt.interest_over_time()
        if "isPartial" in hasil.columns:
            hasil = hasil.drop(columns=["isPartial"])

        if hasil.empty:
            print("Google Trends menjawab, tetapi tidak ada data untuk kata kunci ini.")
            print("Biasanya karena kata kuncinya terlalu jarang dicari, atau salah ketik.")
            print("\nCoba kata kunci lain, atau jalankan sel berikutnya untuk salinan cadangan.")
        else:
            trends = hasil
            print(f"Berhasil. {len(trends)} titik data.")
            display(trends.tail())

    except ModuleNotFoundError:
        print("Paket pytrends belum terpasang di runtime ini.")
        print("Penyebab tersering: sel Langkah 1 belum dijalankan, atau runtime baru di-restart.")
        print("\nDua pilihan, dua-duanya beres:")
        print("  a. Jalankan sel Langkah 1 dulu, lalu jalankan ulang sel ini.")
        print("  b. Atau langsung jalankan sel berikutnya. Salinan cadangan tidak butuh pytrends.")

    except Exception as e:
        print("Google Trends menolak permintaan ini. Ini sering terjadi dan bukan salah kamu.")
        print(f"Pesan aslinya: {e}")
        print("\nJalankan sel berikutnya untuk memakai salinan cadangan.")

### Kalau sel di atas gagal

Google membatasi permintaan otomatis. Kalau seluruh kelas menjalankan sel yang sama dalam waktu
berdekatan, sebagian pasti ditolak.

Jalankan sel di bawah untuk memakai salinan yang sudah diunduh dosen. Isinya sama persis.

*(Kalau sel sebelumnya berhasil, sel ini otomatis melewati dirinya sendiri.)*

In [ ]:
# --- Jalur cadangan: salinan ekspor Google Trends di repositori kelas ---
#
# CATATAN UNTUK DOSEN: ganti USERNAME dan REPO di bawah dengan milik Anda.
# Tautan mentah GitHub bentuknya:
#   https://raw.githubusercontent.com/FEB-Unpad/ds-econ/main/data/trends_phk_lowongan.csv
# Cara tercepat mendapatkannya: buka berkas CSV di GitHub, klik tombol "Raw", salin URL-nya.

URL_CSV = "https://raw.githubusercontent.com/FEB-Unpad/ds-econ/main/data/trends_phk_lowongan.csv"


def baca_csv_trends(url):
    """Membaca berkas ekspor Google Trends apa adanya.

    Empat hal yang perlu ditangani:
    - berkasnya punya dua baris pembuka sebelum baris header
    - kolom tanggal bernama "Month" atau "Week", tergantung panjang periode yang diminta
    - nilai yang sangat kecil ditulis sebagai teks "<1", bukan angka
    - nama kolom berbentuk "PHK: (Indonesia)", perlu dirapikan
    """
    df = pd.read_csv(url, skiprows=2)
    df = df.rename(columns={df.columns[0]: "date"})
    df["date"] = pd.to_datetime(df["date"])
    for kolom in df.columns[1:]:
        df[kolom] = pd.to_numeric(
            df[kolom].astype(str).str.replace("<1", "0.5", regex=False),
            errors="coerce",
        )
    df.columns = ["date"] + [c.split(":")[0].strip() for c in df.columns[1:]]
    return df.set_index("date")


if trends is None:
    trends = baca_csv_trends(URL_CSV)
    print(f"Memakai salinan cadangan dari repositori kelas. {len(trends)} titik data.")
    display(trends.tail())
else:
    print("Tidak perlu. Data langsung dari Google Trends sudah masuk.")

## Langkah 4 — Grafik pertama, apa adanya

Gambar dulu tanpa perlakuan apa pun. Perhatikan baik-baik sebelum menggulir ke bawah.

In [ ]:
fig, ax = plt.subplots()

for kolom in trends.columns:
    ax.plot(trends.index, trends[kolom], linewidth=1.6, label=kolom)

ax.set_ylabel("Indeks Google Trends (0-100)")
ax.legend(frameon=False, loc="upper right")
ax.set_title("Grafik 1 — apa adanya. Di mana garis PHK?",
             loc="left", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("Nilai tertinggi tiap kata kunci:")
print(trends.max().to_string())

**Garis PHK nyaris menempel di dasar grafik.** Nilainya hanya 2 sampai 12, sementara
`lowongan kerja` mencapai 100.

Apakah itu berarti tidak ada yang mencari kata PHK? Tidak. Google Trends menormalkan **seluruh
kata kunci dalam satu permintaan** terhadap satu titik tertinggi yang sama. Karena
`lowongan kerja` jauh lebih sering diketik, `PHK` tergencet sampai tidak terbaca.

Angka indeks Google Trends tidak punya arti absolut. Ia hanya punya arti relatif terhadap isi
permintaan yang kamu ajukan sendiri. Ganti daftar kata kuncinya, angkanya berubah semua.

Perbaikannya sederhana: skalakan setiap seri terhadap puncaknya sendiri. Sekarang kita bisa
membandingkan **bentuk** pergerakannya, bukan tingginya.

In [ ]:
# Setiap seri diskala ke puncaknya sendiri, sehingga bentuknya bisa dibandingkan.
skala = trends / trends.max() * 100

fig, ax1 = plt.subplots()

warna = {"PHK": "#c0392b", "lowongan kerja": "#7f8c8d"}
for kolom in skala.columns:
    ax1.plot(skala.index, skala[kolom], linewidth=1.8,
             label=f"Pencarian: {kolom}", color=warna.get(kolom))
ax1.set_ylabel("Indeks diskala ke puncak masing-masing (=100)")

ax2 = ax1.twinx()
ax2.scatter(bps["periode_data"], bps["tpt_persen"], color="black", zorder=5, s=80,
            label="TPT resmi BPS")
ax2.set_ylabel("TPT (persen)")
ax2.set_ylim(3.5, 8)
ax2.grid(False)

for _, r in bps.iterrows():
    ax2.annotate("", xy=(r["tanggal_rilis"], r["tpt_persen"]),
                 xytext=(r["periode_data"], r["tpt_persen"]),
                 arrowprops=dict(arrowstyle="->", linestyle="--", color="firebrick", lw=1.1))
    ax2.text(r["tanggal_rilis"], r["tpt_persen"], f"  terbit {r['tanggal_rilis']:%b %Y}",
             fontsize=8, color="firebrick", va="center")

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper right", frameon=False, fontsize=9)
ax1.set_title("Grafik 2 — sinyal PHK melonjak April 2020. Angka resminya terbit 5 November 2020.",
              loc="left", fontsize=12.5, fontweight="bold")
plt.tight_layout()
plt.show()

## Yang baru saja kamu lihat

**Satu kata kunci bekerja, satu lagi gagal, dan keduanya sama-sama masuk akal di awal.**

**PHK.** Rata-rata sepanjang 2019 hanya sekitar 2. Pada April 2020 melonjak ke 12, lebih dari
lima kali lipat, tepat ketika gelombang pemutusan hubungan kerja dimulai. Titik hitam di
dekatnya adalah TPT Agustus 2020 sebesar 7,07 persen. Panah merahnya menunjuk ke November 2020,
saat angka itu baru diumumkan. Sinyalnya menyala tujuh bulan lebih dulu.

**lowongan kerja.** Meluruh terus dari 99 pada awal 2019 menjadi 17 pada pertengahan 2026,
turun sekitar delapan puluh persen. Sementara itu TPT justru membaik tipis. Kalau kata kunci ini
benar-benar mengukur pengangguran, Indonesia mestinya sudah mencapai lapangan kerja penuh.
Jelas tidak.

Penyebabnya bukan pasar kerja, melainkan perpindahan perilaku. Orang berhenti mencari kerja
lewat Google dan pindah ke aplikasi lowongan, LinkedIn, dan media sosial. Yang berubah bukan
jumlah penganggur, melainkan cara mereka mencari.

**Pelajarannya.** Sumber datanya sama, alatnya sama, periodenya sama. Yang membedakan hanya satu
hal: apakah kamu paham bagaimana angka itu terbentuk, dan apakah proses pembentukannya berubah
selama periode yang kamu amati. Inilah kegagalan Google Flu Trends yang dibahas di Materi 1,
terjadi pada data Indonesia, dan barusan kamu lihat sendiri.

---

**Dua catatan sebelum kamu melangkah lebih jauh.**

**Garis PHK terlihat kotak-kotak setelah 2021.** Itu bukan gejolak pasar kerja. Google Trends
memberi bilangan bulat, dan di luar masa pandemi nilai PHK hanya berkisar 2 sampai 5. Setelah
diskalakan ke puncaknya sendiri, selisih satu angka saja meloncat sekitar delapan poin. Yang
kamu lihat adalah batas ketelitian datanya, bukan sinyal ekonomi. Pelajaran praktisnya: kalau
angka aslinya kecil, penskalaan membesarkan deraunya sekaligus sinyalnya.

**Sebelum kamu terlalu senang dengan PHK.** Lonjakan April 2020 bisa saja mencerminkan orang
membaca berita tentang PHK, bukan orang yang mengalaminya. Dan satu episode bukan bukti. Kita
kembali ke soal ini di Langkah 6.

## Langkah 5 — Sekarang giliran kamu

Ubah tiga hal di sel Langkah 3, lalu jalankan ulang dari sana.

1. **Ganti kata kunci.** Ganti isi `KATA`. Pilih dua sampai lima kata yang menurutmu menandai
   tekanan di pasar kerja Indonesia. Contoh yang bisa dicoba: `"kartu prakerja"`, `"pesangon"`,
   `"resign"`, `"loker"`, `"bansos"`.
2. **Ganti periode.** Ubah `PERIODE`. Kalau kamu persempit ke bawah lima tahun, Google berpindah
   dari data bulanan ke mingguan. Coba `"2022-01-01 2026-08-23"` dan lihat bedanya.
3. **Ganti wilayah.** Ubah `WILAYAH`. Gunakan `"ID-JB"` untuk Jawa Barat, `"ID-JK"` untuk DKI
   Jakarta, `"ID-BT"` untuk Banten. Bandingkan bentuknya dengan angka nasional.

Setelah puas, jalankan sel di bawah untuk menyimpan grafikmu.

In [ ]:
plt.savefig("grafik_sesi1.png", dpi=150, bbox_inches="tight")
from google.colab import files
files.download("grafik_sesi1.png")
print("Grafik terunduh. Lampirkan di pengumpulan tugas kamu.")

## Langkah 6 — Empat pertanyaan sebelum kamu percaya grafik ini

Jawab singkat di sel teks di bawah. Jawaban kamu ikut dikumpulkan dan dinilai.

**Pertanyaan 1.** Di Grafik 1 garis PHK hampir tidak terlihat, di Grafik 2 ia melonjak
tinggi. Datanya sama persis. Jelaskan apa yang berubah, lalu sebutkan satu kesalahan
kesimpulan yang mungkin diambil orang yang hanya melihat Grafik 1.

**Pertanyaan 2.** Siapa yang tidak muncul dalam data pencarian ini? Sebutkan satu kelompok
pekerja Indonesia yang kemungkinan besar kehilangan pekerjaan pada 2020 tetapi tidak tercatat
mengetik "PHK" di Google. Apa akibatnya bagi kesimpulan yang boleh kamu tarik?

**Pertanyaan 3.** Lonjakan PHK April 2020 bisa berarti banyak orang di-PHK, atau bisa berarti
banyak orang membaca berita tentang PHK. Kedua penjelasan itu menghasilkan grafik yang sama.
Data tambahan apa yang kamu perlukan untuk membedakan keduanya?

**Pertanyaan 4.** Kata kunci `PHK` bekerja, `lowongan kerja` tidak. Andaikan seseorang mencoba
dua puluh kata kunci lalu melaporkan satu yang paling cocok dengan TPT. Apa nama masalah ini,
dan kenapa temuan yang dihasilkan cara begitu tidak layak dipercaya, meskipun grafiknya rapi?

### Jawaban kamu

*(klik dua kali di sini untuk mengetik)*

**1.**

**2.**

**3.**

**4.**

## Yang perlu kamu bawa ke Sesi 2

Notebook ini sengaja berhenti sebelum membangun model, dan alasannya ada di pertanyaan ketiga
dan keempat.

Kita punya satu episode yang meyakinkan, yaitu April 2020. Satu episode bukan bukti hubungan
yang bisa diandalkan. Ditambah lagi, TPT hanya terbit dua kali setahun, jadi sejak 2015 hanya
ada sekitar dua puluh pasang pengamatan untuk diuji. Regresi dengan dua puluh titik tidak
membuktikan prediksi.

Peneliti yang serius menangani ini dengan metode frekuensi campuran, yang menggabungkan data
bulanan dan semesteran tanpa membuang informasi. Ada karya Indonesia tentang persis ini di
*Journal of Developing Economies* Universitas Airlangga. Kita bahas di Sesi 7 dan 8.

Sampai saat itu, tahan diri untuk tidak menyimpulkan lebih dari yang ditopang data. Kelas ini
menilai kejujuran metodologis lebih tinggi daripada temuan yang mengesankan.

---

### Yang wajib kamu kumpulkan dari lab ini
1. Tautan notebook Colab kamu, dengan akses "siapa saja yang memiliki tautan dapat melihat"
2. Berkas `grafik_sesi1.png` hasil modifikasimu, bukan grafik bawaan
3. Jawaban empat pertanyaan di Langkah 6

### Kalau tersendat
Tulis di Forum Bantuan Teknis hari ini juga. Jangan menunggu sampai Sesi 3.